In [ ]:
import numpy as np
import pandas as pd 
import pickle

from pathlib import Path
from astropy.timeseries import LombScargle
from scipy.stats import skew, kurtosis, shapiro
from utils.preprocessing import fourier_features, stetson_K, fourier_fit

In [ ]:
def compute_features_to_csv(light_curves, outfile="features.csv"):
    """
    light_curves: list of (t, f) tuples OR (star_id, t, f) triples
    outfile     : CSV file to write

    Returns the DataFrame of features.
    """

    rows = []

    for idx, item in enumerate(light_curves):
        # Support either (t, f) or (star_id, t, f)
        if isinstance(item, (list, tuple)) and len(item) == 3:
            star_id, t, f = item
        else:
            t, f = item
            star_id = idx 

        # remove NaNs
        mask = np.isfinite(t) & np.isfinite(f)
        t, f = t[mask], f[mask]

        # --- Period (Lomb–Scargle) ---
        freq, power = LombScargle(t, f).autopower()
        best_period = 1 / freq[np.argmax(power)]

        # --- Flux distribution features ---
        Q1 = np.percentile(f, 25)
        Q3 = np.percentile(f, 75)
        Q31 = Q3 - Q1
        Std = np.std(f)
        gamma1 = skew(f)
        gamma2 = kurtosis(f, fisher=True)
        W, _ = shapiro(f)
        K = stetson_K(f)

        # --- Fourier features ---
        R21, R31, phi21, phi31, Amp = fourier_features(best_period, t, f)

        # store (ensure star_id is first key)
        rows.append({
            "star_id": star_id,
            "period": best_period,
            "Q31": Q31,
            "Amp": Amp,
            "W": W,
            "K": K,
            "Std": Std,
            "gamma1": gamma1,
            "gamma2": gamma2,
            "R21": R21,
            "R31": R31,
            "phi21": phi21,
            "phi31": phi31
        })

        print(f"Processed light curve {idx+1}/{len(light_curves)}")

    df = pd.DataFrame(rows)
    # Reorder columns explicitly to keep star_id first
    ordered_cols = [
        "star_id", "period", "Q31", "Amp", "W", "K", "Std",
        "gamma1", "gamma2", "R21", "R31", "phi21", "phi31"
    ]
    df = df[ordered_cols]
    df.to_csv(outfile, index=False)
    print(f"\nSaved feature table to: {outfile}")

    return df

In [ ]:
# Load list of (time, flux) tuples
pkl_path = Path("/home/admin/main/ucsd-phys-139-final/data/time_flux_pdcsap.pkl")
if not pkl_path.exists():
    raise FileNotFoundError(f"Pickle not found: {pkl_path}")

with pkl_path.open("rb") as f:
    light_curves = pickle.load(f)

print(f"Loaded {len(light_curves)} light curves from {pkl_path}")

# Compute features and save to CSV
out_csv = Path("/home/admin/main/ucsd-phys-139-final/data/features.csv")
df = compute_features_to_csv(light_curves, outfile=str(out_csv))
print(f"Saved {len(df)} rows to {out_csv}")

ModuleNotFoundError: No module named 'numpy._core.numeric'